# Agente Strands com Observabilidade Instana no Amazon Bedrock AgentCore Runtime

## Visão Geral

Este notebook demonstra como implantar um agente Strands no Amazon Bedrock AgentCore Runtime com integração de observabilidade Instana. A implementação utiliza modelos Amazon Bedrock e envia dados de telemetria para o Instana através do OpenTelemetry (OTEL).

## Componentes Principais

- **Strands Agents**: Framework Python para construir agentes baseados em LLM com suporte a telemetria integrado
- **Amazon Bedrock AgentCore Runtime**: Serviço de runtime gerenciado para hospedar e escalar agentes na AWS
- **Instana**: Observabilidade em tempo real e monitoramento de desempenho para aplicações que recebem traces via OTEL
- **OpenTelemetry**: Protocolo padrão da indústria para coleta e exportação de dados de telemetria

## Arquitetura

O agente é containerizado e implantado no AgentCore Runtime, que fornece endpoints HTTP para invocação. Os dados de telemetria fluem do agente Strands através dos exporters OTEL para o Instana para monitoramento e depuração. A implementação desabilita a observabilidade padrão do AgentCore para utilizar o Instana em seu lugar.

## Pré-requisitos

- Python 3.10+
- [Comece a usar o Amazon Bedrock AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-get-started-toolkit.html#agentcore-get-started-prerequisites)
- Credenciais AWS configuradas com permissões do Bedrock e [AgentCore](https://docs.aws.amazon.com/aws-managed-policy/latest/reference/BedrockAgentCoreFullAccess.html)
- Conta [Instana](https://www.ibm.com/products/instana)
- Docker instalado localmente
- Acesso aos modelos Amazon Bedrock

### Encontrando o Endpoint Instana Correto

Para encontrar o endpoint OTLP correto para sua instância Instana:

1. Abra a **barra lateral** no Instana.
2. Role até o final e selecione **About Instana**.
3. Observe a **Instance Region** exibida.
4. Com base nessa região, escolha o **endpoint HTTP OTLP (4318)** apropriado usando a [documentação](https://www.ibm.com/docs/en/instana-observability/1.0.309?topic=instana-backend) de endpoints


### Obtendo sua Chave Instana

Para obter sua chave Instana:

1. Na barra lateral, clique em **Agents & Collectors**.
2. Selecione **Linux – Automatic Installation (One-liner)**.
3. Copie a chave que aparece após o flag `-a` — esta é sua **Agent Key**, que é usada como **Instana Key**.

Consulte a imagem abaixo para orientação.

<div style="text-align:left">
    <img src="../images/instana_agent_key.png" width="99%"/>
</div>

### Crie um novo arquivo chamado `.env` no diretório raiz do seu projeto e adicione as seguintes variáveis de ambiente:

Execute a célula abaixo para gerar automaticamente um arquivo .env na raiz do seu projeto. Após a criação do arquivo, atualize os valores de placeholder com seu endpoint OTLP e chave de API reais do Instana.

O arquivo `.env` é usado para armazenar com segurança seus detalhes de configuração (como chaves de API e endpoints) para que possam ser facilmente carregados na sua aplicação sem codificá-los diretamente.

**Nota**: Nunca faça commit do arquivo `.env` no GitHub ou compartilhe sua chave Instana publicamente. Adicione `.env` ao seu arquivo `.gitignore` para manter as credenciais seguras.

### Exemplo de conteúdo do arquivo .env:

In [ ]:
%%writefile .env

# Instana OTLP endpoint (e.g., https://otlp-blue-saas.instana.io:4318)
OTEL_EXPORTER_OTLP_ENDPOINT="<instana_endpoint>"

# Note: This example uses StrandsTelemetry() for exporting, which supports only the HTTP OTLP endpoint.
# Make sure to use the HTTP endpoint when working with Strands Telemetry.

# Instana key
INSTANA_KEY="<agent_key>"

Instale as dependências executando a célula abaixo:

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Configurar Credenciais AWS

Configure suas credenciais AWS seguindo os links fornecidos na seção `Pré-requisitos` antes de executar o notebook.

## Implementação do Agente

O arquivo do agente (`strands_nova.py`) implementa um agente de viagens com capacidades de busca na web. A configuração principal inclui:
- Inicialização da telemetria Strands

In [ ]:
%%writefile strands_nova.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Function to initialize Bedrock model
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    model_id = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Initialize the Bedrock model
bedrock_model = get_bedrock_model()

# Define the agent's system prompt
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """Initialize the agent with proper telemetry configuration."""

    # Initialize Strands telemetry with 3P configuration
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    
    # Create and cache the agent
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # Initialize agent with proper configuration
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

### Configurar implantação no AgentCore Runtime

Em seguida, usaremos o starter toolkit para configurar a implantação no AgentCore Runtime.
Durante esta etapa, você configurará o entrypoint, a execution role que você criou anteriormente e o arquivo de requirements.
Você também configurará o starter kit para criar automaticamente o repositório Amazon ECR quando ele for lançado.

Quando você executar o comando de configuração, um Dockerfile será gerado automaticamente com base no código da sua aplicação.

**Nota:**
O bedrock_agentcore_starter_toolkit habilita a Observabilidade do AgentCore por padrão.
Se você planeja usar o Instana para observabilidade, precisará remover a configuração de Observabilidade padrão do AgentCore, conforme descrito na próxima seção.

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_instana_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_nova.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response

### Configuração Alternativa (Usando uma IAM Role Pré-Criada)

Se sua conta AWS **não permite a criação automática de roles** ou se você deseja **mais controle sobre as permissões**, você pode usar uma **IAM role pré-criada** em vez de deixar o toolkit criar uma automaticamente.

Neste caso:
- Defina `auto_create_execution_role=False`
- Forneça o ARN da sua IAM role existente através do parâmetro `execution_role`.

Esta abordagem é útil quando:
- Sua organização aplica **políticas IAM de menor privilégio**.
- Você deseja **reutilizar uma execution role comum** em múltiplas implantações do AgentCore.
- Você está trabalhando em um **ambiente restrito** (por exemplo, contas AWS empresariais ou compartilhadas) onde a criação de novas roles não é permitida.

Consulte o exemplo abaixo para ver como configurar o runtime com uma IAM role gerenciada manualmente.

In [ ]:
# Alternative Configuration (Optional)

from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

ROLE_ARN = "your_IAM_role"

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_instana_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_nova.py",
    auto_create_execution_role=False,   # This disables the auto-create role
    execution_role=ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_instana_observability",
    disable_otel=True,
)
response

## Implantar no AgentCore Runtime

Agora que seu Dockerfile foi gerado, é hora de lançar seu agente no AgentCore Runtime.

Esta etapa irá automaticamente:
- Criar o repositório Amazon ECR (se ainda não existir)
- Implantar seu agente containerizado no ambiente AgentCore Runtime
- Carregar suas variáveis de ambiente (como o endpoint e a chave de API do Instana) do arquivo `.env` que você criou anteriormente

**NOTA:** Certifique-se de que seu arquivo `.env` esteja corretamente configurado antes de executar esta célula — caso contrário, os dados de telemetria não serão exportados para o Instana.

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Fetch configuration
otel_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT")
instana_key = os.getenv("INSTANA_KEY")

# Format Instana header
otel_auth_header = f"x-instana-key={instana_key}"

# Launch the AgentCore runtime
launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "amazon.nova-lite-v1:0",
        "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,
        "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,
        "OTEL_SERVICE_NAME": "AWS-APP",
        "OTEL_EXPORTER_OTLP_INSECURE": "false",
        "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)

launch_result

## Verificar Status da Implantação

Após o lançamento da implantação, pode levar alguns minutos para o AgentCore Runtime concluir a configuração.
Você pode usar o código a seguir para monitorar o status da implantação em tempo real.

Este trecho verifica continuamente o status do seu endpoint AgentCore a cada poucos segundos até que ele alcance um estado final — como:
- READY → Implantação concluída com sucesso
- CREATE_FAILED, UPDATE_FAILED ou DELETE_FAILED → A implantação encontrou um erro

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invocando o AgentCore Runtime

Finalmente, podemos invocar o agente implantado com um prompt de exemplo para testar sua resposta e verificar se está funcionando conforme esperado.

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "I'm planning a weekend trip to london. What are the must-visit places and local food I should try?"})

Use o código abaixo para exibir de forma organizada a saída do agente.

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

## Visualizar Traces no Instana

Para visualizar os traces:
1. Acesse seu painel do Instana
2. Clique em `Analytics` na barra lateral.
3. Marque a caixa `Show internal calls` na aba `Hidden calls`.
4. Clique em `Add Filter` e selecione `Service Name`
5. Pesquise por "strands-agents" na barra de busca.
6. Clique para visualizar e analisar a lista de chamadas associadas. Cada chamada representa uma interação do usuário.
7. Clique em qualquer chamada para ver os dados completos do trace.

Os traces incluirão:
- Detalhes de invocação do agente
- Chamadas de ferramentas (busca na web)
- Interações com o modelo com latência e uso de tokens
- Payloads de requisição/resposta

## Limpeza (Opcional)

Após concluir os testes, use o código abaixo para excluir o AgentCore Runtime e o repositório Amazon ECR associado para evitar uso desnecessário de recursos e custos.

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Resumo

Você implantou com sucesso um agente Strands no Amazon Bedrock AgentCore Runtime com observabilidade Instana. A implementação demonstra:
- Integração de agentes Strands com o AgentCore Runtime
- Configuração do OpenTelemetry para enviar traces ao Instana
- Ordem de inicialização adequada para garantir a configuração de telemetria

O agente agora está rodando em um ambiente gerenciado e escalável com observabilidade completa através do Instana.